### Silver to Gold — HR Domain (dim_broker)
**Author:** Virendra Tambavekar  
**Task:** Build Gold dimension table `dim_broker` from Silver  
**Domain:** HR (Broker)  
**Pipeline Stage:** Silver → Gold  
**Source:** `charles_schwab_retailbrokerage_dev_team_lemma.silver.broker` (50,000 rows)  
**Target:** `charles_schwab_retailbrokerage_dev_team_lemma.gold.dim_broker` (14,239 rows)  

**Transformations:**
- Filter: `JOB_CODE = 314` (brokers only)
- Surrogate Key: `SK_BrokerID = EMPLOYEE_ID`
- SCD-1 dimension: `IsCurrent = TRUE`, `EffectiveDate = batch_date`, `EndDate = 9999-12-31`
- No history tracking — always current state only

In [0]:
#Importing required libraries
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import LongType

#Intializing Logger
logger = logging.getLogger("SilverToGoldBroker")
logger.setLevel(logging.INFO)

In [0]:
source_silver_table = "charles_schwab_retailbrokerage_dev_team_lemma.silver.broker"
target_gold_table = "charles_schwab_retailbrokerage_dev_team_lemma.gold.dim_broker"

In [0]:
# Idempotency Logic

def idempotency_check(table_name):
    #Check if target table exists and log state before overwrite.
    if spark.catalog.tableExists(table_name):
        existing_count = spark.table(table_name).count()
        logger.info(f"Idempotency: Target table '{table_name}' already exists with {existing_count} rows. Will be overwritten.")
    else:
        logger.info(f"Idempotency: Target table '{table_name}' does not exist. Will be created.")

idempotency_check(target_gold_table)

In [0]:
def build_dim_broker():
    logger.info("Building dim_broker")

    try:
        #Read silver table
        silver_df = spark.read.table(source_silver_table)
        #Filter Brokers (Job Code 314)
        brokers_df = silver_df.filter(col("JOB_CODE") == 314)
        #SCD-1 Logic : Surrogate Key and Renaming the columns
        gold_df = (
            brokers_df
            .withColumn("SK_BrokerID",col("EMPLOYEE_ID").cast(LongType()))
            .withColumn("BrokerID",col("EMPLOYEE_ID").cast(LongType()))
            .withColumn("ManagerID",col("MANAGER_ID").cast(LongType()))
            .withColumn("LastName",col("LAST_NAME"))
            .withColumn("FirstName",col("FIRST_NAME"))
            .withColumn("MiddleInitial",col("MIDDLE_INITIAL"))
            .withColumn("Branch",col("BRANCH_ID"))
            .withColumn("Office",col("OFFICE"))
            .withColumn("Phone",col("PHONE"))
            .withColumn("IsCurrent",lit(True))
            #.withColumn("_batch", )
            .withColumn("_batch", col("_batch_id"))
            .withColumn("EffectiveDate",current_date())
            .withColumn("EndDate",to_date(lit("9999-12-31")))
        )

        final_gold_df = gold_df.select(
            "SK_BrokerID",
            "BrokerID",
            "ManagerID",
            "FirstName",
            "LastName",
            "MiddleInitial",
            "Branch",
            "Office",
            "Phone",
            "IsCurrent",
            "_batch",
            "EffectiveDate",
            "EndDate"
        )
        #Write Gold Table
        (
            final_gold_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_gold_table)
        )
        logger.info("Transformed and wrote dim_broker")
        return True
    except Exception as e:
        logger.error(f"Erorr in transforming {str(e)}")
        raise e
table_built = build_dim_broker()

In [0]:
if table_built:
    try:
        #Read gold table to validate
        gold_df = spark.table(target_gold_table)
        actual_count  = gold_df.count()
        expected_count  = 14239

        logger.info("Reconciliation Summary")
        logger.info("Target Table : dim_broker")
        logger.info(f"Expected Count: {expected_count}")
        logger.info(f"Actual Count : {actual_count}")
        display(actual_count)

        if actual_count == expected_count:
            logger.info("Reconciliation Successful")
        else:
            logger.info("Reconciliation Failed")
            raise Exception ("Gold Reconciliation Failed")
        display(gold_df.limit(10))
    except Exception as e:
        logger.error(f"Error in reconciliation : {str(e)}")
        raise e